# Attention Mechanism Acceleration - Colab Runner

**Run this 3 times** with different runtimes to collect real hardware data:
1. Runtime → **CPU** → Run All → downloads `results_cpu.zip`
2. Runtime → **T4 GPU** → Run All → downloads `results_gpu.zip`
3. Runtime → **TPU** → Run All → downloads `results_tpu.zip`

The notebook auto-detects hardware and runs ONLY the relevant paradigm on real hardware.
Simulations (GaAs, CdTe, Dataflow, etc.) are run locally — not here.

## Step 1: Upload project zip

In [ ]:
from google.colab import files
import zipfile, os, shutil

# Always start from /content/ to avoid "directory deleted under us" errors
os.chdir('/content/')

# Clean old copy to ensure fresh code
if os.path.exists('/content/attention_mechanism_acceleration'):
    shutil.rmtree('/content/attention_mechanism_acceleration')

uploaded = files.upload()
zip_name = list(uploaded.keys())[0]
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('/content/')
os.chdir('/content/attention_mechanism_acceleration')
print(f'Ready: {os.getcwd()}')

## Step 2: Install, detect, run, download
Single cell — does everything automatically.

In [ ]:
import subprocess, sys, shutil, os

# --- Verify we're in the right directory ---
if not os.path.exists('config.py'):
    raise RuntimeError("ERROR: config.py not found! Run Step 1 (upload zip) first.")

# --- Print version ---
sys.path.insert(0, os.getcwd())
import config
print(f'Code version: {config.BUILD_VERSION}')

# --- Install base dependencies ---
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'numpy', 'matplotlib', 'pandas', 'tabulate'])

# --- Install torch only if not already available ---
try:
    import torch
except ImportError:
    print('torch not found, installing...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'torch'])
    import torch

# --- Detect hardware ---
hw_type = 'cpu'
hw_detail = 'Colab CPU'

if torch.cuda.is_available():
    hw_type = 'gpu'
    hw_detail = torch.cuda.get_device_name(0)
else:
    # TPU detection via JAX (native TPU framework on Colab)
    tpu_detected = False

    # Method 1: JAX (pre-installed on Colab TPU runtime)
    try:
        import jax
        tpu_devices = jax.devices("tpu")
        if len(tpu_devices) > 0:
            tpu_detected = True
            hw_detail = f"JAX: {tpu_devices[0]}"
            print(f'TPU detected via JAX: {tpu_devices}')
    except Exception as e:
        print(f'JAX TPU check: {e}')

    # Method 2: Try installing JAX with TPU support if not found
    if not tpu_detected:
        try:
            print('Trying to install JAX with TPU support...')
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                                   'jax[tpu]', '-f',
                                   'https://storage.googleapis.com/jax-releases/libtpu_releases.html'],
                                  stderr=subprocess.DEVNULL)
            import jax
            tpu_devices = jax.devices("tpu")
            if len(tpu_devices) > 0:
                tpu_detected = True
                hw_detail = f"JAX: {tpu_devices[0]}"
                print(f'TPU detected after JAX install: {tpu_devices}')
        except Exception as e:
            print(f'JAX TPU install failed: {e}')

    if tpu_detected:
        hw_type = 'tpu'

print(f'\n{"="*60}')
print(f'HARDWARE DETECTED: {hw_type.upper()} ({hw_detail})')
print(f'PyTorch: {torch.__version__}')
if torch.cuda.is_available():
    print(f'CUDA: {torch.version.cuda}')
    print(f'GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
if hw_type == 'tpu':
    try:
        import jax
        print(f'JAX: {jax.__version__} (backend: {jax.default_backend()})')
    except Exception:
        pass
print(f'{"="*60}')
print(f'\nRunning ONLY real-hardware paradigms for {hw_type.upper()}...')
print(f'(Simulations are run locally, not here)\n')

# --- Run benchmarks IN-PROCESS ---
import runpy
sys.argv = ['run_all.py', 'real', hw_type]
runpy.run_path('run_all.py', run_name='__main__')

# --- Show results ---
import pandas as pd
df = pd.read_csv('results/scaling_results.csv')

print(f'\n{"="*60}')
print(f'RESULTS — Real {hw_type.upper()} Measurements')
print(f'{"="*60}')

summary = df[df['seq_len'] == df['seq_len'].max()]
cols = ['paradigm_short', 'embed_dim', 'total_effective_time_s',
        'throughput_gflops', 'info_execution_mode']
cols = [c for c in cols if c in df.columns]
summary = summary[cols].sort_values('total_effective_time_s')
print(summary.to_string(index=False))

# --- Download results labeled by hardware ---
output_name = f'/content/results_{hw_type}'
shutil.make_archive(output_name, 'zip', 'results')

from google.colab import files as colab_files
colab_files.download(f'{output_name}.zip')
print(f'\n>>> Downloaded: results_{hw_type}.zip')

## (Optional) View plots

In [ ]:
from IPython.display import Image, display
import os

for plot in sorted(f for f in os.listdir('results') if f.endswith('.png')):
    print(f'\n--- {plot} ---')
    display(Image(filename=f'results/{plot}', width=700))